In [1]:
import random
from autogen import ConversableAgent, Agent
from autogen.agentchat.contrib.web_surfer import WebSurferAgent, UserProxyAgent  # noqa: E402
from dotenv import load_dotenv
import os
load_dotenv('.env')
config_list = [
  {
    "model": "gpt-35-turbo",
    "api_type": "azure",
    "api_key": os.environ['AZURE_OPENAI_API_KEY'],
    "base_url": os.environ['AZURE_OPENAI_ENDPOINT_TURBO35'],
    "api_version": "2023-03-15-preview"
  }
]
llm_config = {
    "config_list": config_list,
   "timeout": 600,
    "cache_seed": 44

}

# def print_messages(recipient, messages, sender,config):

#     #chat_interface.send(messages[-1]['content'], user=messages[-1]['name'], avatar=avatar[messages[-1]['name']], respond=False)
#     print(f"Messages from: {sender.name} sent to: {recipient.name} | num messages: {len(messages)} | message: {messages[-1]}")

#     return False, None

def send_messages(recipient, messages, sender,config):

    #chat_interface.send(messages[-1]['content'], user=messages[-1]['name'], avatar=avatar[messages[-1]['name']], respond=False)
    msg=f"Messages from: {sender.name} sent to: {recipient.name} | num messages: {len(messages)} | message: {messages[-1]}"

    return msg, None

num=random.randint(1, 100)
agent_with_number = ConversableAgent(
    "agent_with_number",
    system_message="You are playing a game of guess-my-number. You have the "
    f"number {num} in your mind, and I will try to guess it. "
    "If I guess too high, say 'too high', if I guess too low, say 'too low'. ",
    llm_config=llm_config,
    max_consecutive_auto_reply=1,
    is_termination_msg=lambda msg: f"{num}" in msg["content"],  # terminate if the number is guessed by the other agent
    human_input_mode="NEVER",  # never ask for human input
)

human_proxy = UserProxyAgent(
    name="Admin",
    human_input_mode="ALWAYS",
    code_execution_config=False,
)

human_proxy.register_reply(
    [Agent, None],
    reply_func=send_messages, 
    config={"callback": None},
)



### RAG

In [10]:
from chromadb.utils import embedding_functions

openai_embedding_function = embedding_functions.OpenAIEmbeddingFunction(api_key = os.getenv("AZURE_OPENAI_API_KEY"))

In [7]:
config_list

[{'model': 'gpt-35-turbo',
  'api_type': 'azure',
  'api_key': 'REDACTED_AZURE_KEY',
  'base_url': 'https://REDACTED_AZURE_ENDPOINT/openai/deployments/gpt-35-turbo/chat/completions?api-version=2023-03-15-preview',
  'api_version': '2023-03-15-preview'}]

In [15]:
from autogen.agentchat.contrib.retrieve_assistant_agent import RetrieveAssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent
import chromadb
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", "\r", "\t"])

assistant = RetrieveAssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="ALWAYS",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "code",
        "docs_path": [
            "http://www.buffalobicycle.com/storage/documents/wbr_bicycle_maintenance_manual.pdf"
        ],
        "custom_text_types": ["mdx"],
        "chunk_token_size": 2000,
        "model": config_list[0]["model"],
        "client": chromadb.PersistentClient(path="/tmp"),
        "embedding_model": openai_embedding_function,
        "get_or_create": True,  # set to False if you don't want to reuse an existing collection, but you'll need to remove the collection manually
        "custom_text_split_function": text_splitter.split_text,
    },
    code_execution_config=False,  # set to False if you don't want to execute the code
)

In [16]:
ragproxyagent.initiate_chat(assistant, problem="What is the workflow in docGPT?", n_results=2)

ragproxyagent (to assistant):



--------------------------------------------------------------------------------
assistant (to ragproxyagent):

How can I assist you today?

--------------------------------------------------------------------------------
ragproxyagent (to assistant):

tell me more

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

I'd be happy to help, but I'll need a bit more information. What specific topic or question would you like to know more about?

--------------------------------------------------------------------------------
ragproxyagent (to assistant):

the bike

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

Bikes are a popular mode of transportation and recreation for people all over the world. They come in various styles, such as road bikes, mountain bikes, hybrid bikes, and more, each designed for different purposes and terrain

ChatResult(chat_id=None, chat_history=[{'content': '', 'role': 'assistant'}, {'content': 'How can I assist you today?', 'role': 'user'}, {'content': 'tell me more', 'role': 'assistant'}, {'content': "I'd be happy to help, but I'll need a bit more information. What specific topic or question would you like to know more about?", 'role': 'user'}, {'content': 'the bike', 'role': 'assistant'}, {'content': 'Bikes are a popular mode of transportation and recreation for people all over the world. They come in various styles, such as road bikes, mountain bikes, hybrid bikes, and more, each designed for different purposes and terrains. Is there anything specific you would like to know about bikes?', 'role': 'user'}, {'content': "can you tell me about the buffalo bike'", 'role': 'assistant'}, {'content': 'The Buffalo Bike is a specific type of bicycle designed by World Bicycle Relief for people in rural and developing areas. It is a sturdy, heavy-duty bicycle designed to carry heavy loads and wit

In [4]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.


In [24]:
os.path.join(os.path.abspath("buffalo_bikes.pdf"))

'c:\\Users\\silas\\Projects\\Telegram\\b'

In [ ]:
assistant.reset()
rag_agent.initiate_chat(assistant, problem="What is the workflow in docGPT?", n_results=2)

In [2]:
human_proxy.get_human_input()

TypeError: ConversableAgent.get_human_input() missing 1 required positional argument: 'prompt'

In [16]:

guess=input("Ready to play?")
result = human_proxy.initiate_chat(
    agent_with_number,  # this is the same agent with the number as before
    message=f"{guess}",
)
# response = await assistant.receive(user_proxy, user_response)


Admin (to agent_with_number):

30

--------------------------------------------------------------------------------
agent_with_number (to Admin):

too low

--------------------------------------------------------------------------------


In [20]:
result.chat_history[-1]['content']

'too low'

In [ ]:
import autogen
from user_proxy_webagent import UserProxyWebAgent
import asyncio

class AutogenChat():
    def __init__(self, chat_id=None, websocket=None):
        self.websocket = websocket
        self.chat_id = chat_id
        self.client_sent_queue = asyncio.Queue()
        self.client_receive_queue = asyncio.Queue()

        self.assistant = autogen.AssistantAgent(
            name="assistant",
            llm_config=llm_config_assistant,
            system_message="""You are a helpful assistant, help the user find the status of his order. 
            Only use the tools provided to do the search. Only execute the search after you have all the information needed. 
            When you ask a question, always add the word "BRKT"" at the end.
            When you responde with the status add the word TERMINATE"""
        )
        self.user_proxy = UserProxyWebAgent(  
            name="user_proxy",
            human_input_mode="ALWAYS", 
            max_consecutive_auto_reply=10,
            is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
            code_execution_config=False,
            function_map={
                "search_db": self.search_db
            }
        )

        # add the queues to communicate 
        self.user_proxy.set_queues(self.client_sent_queue, self.client_receive_queue)

    async def start(self, message):
        await self.user_proxy.a_initiate_chat(
            self.assistant,
            clear_history=True,
            message=message
        )

In [11]:
human_proxy.chat_messages

defaultdict(list,
            {<autogen.agentchat.conversable_agent.ConversableAgent at 0x20cff58b6d0>: [{'content': 'yes',
               'role': 'assistant'}]})

In [4]:
human_proxy.receive(agent_with_number,"10")

TypeError: 'ConversableAgent' object is not iterable

In [1]:
human_proxy.reset()

NameError: name 'human_proxy' is not defined

In [12]:
import asyncio
import nest_asyncio

# Apply nest_asyncio to allow nested event loops
nest_asyncio.apply()
from autogen.io.console import IOConsole
from autogen.io.base import InputStream, OutputStream

# Create an instance of IOConsole for handling input and output
console = IOConsole()
console.print("here")
# async def main():
#     # Define the message to be sent
#     initial_message = "Hello! How can I assist you today?"

#     # Output the initial message and await the user's response asynchronously
#     console.print(f"Agent: {initial_message}")
#     user_response = await console.input("User: ")
#     console.print(f"User: {user_response}")

#     # Send the user response to the assistant agent
#     response = await assistant.receive(user_proxy, user_response)
#     console.print(f"Agent: {response}")

# if __name__ == "__main__":
#     asyncio.run(main())

here


In [5]:
console.input("User: ")

''

In [70]:
human_proxy.send("too high",agent_with_number)

Admin (to agent_with_number):

too high

--------------------------------------------------------------------------------
agent_with_number (to Admin):

85

--------------------------------------------------------------------------------
Admin (to agent_with_number):

10

--------------------------------------------------------------------------------


In [72]:
human_proxy.receive("10",agent_with_number)

agent_with_number (to Admin):

10

--------------------------------------------------------------------------------
Admin (to agent_with_number):

"no"

--------------------------------------------------------------------------------
agent_with_number (to Admin):

too low

--------------------------------------------------------------------------------
Admin (to agent_with_number):

30

--------------------------------------------------------------------------------


<coroutine object ConversableAgent.a_receive at 0x000002AD85877540>

In [57]:
human_proxy.clear_history()

In [68]:
human

AttributeError: 'ConversableAgent' object has no attribute 'chat_history'

In [ ]:


web_surfer = WebSurferAgent(
    "web_surfer",
    llm_config=llm_config,
    summarizer_llm_config=llm_config,
    browser_config={"viewport_size": 4096, "bing_api_key": os.environ["BING_API_KEY"]},
)

user_proxy = UserProxyAgent(
    "user_proxy",
    human_input_mode="NEVER",
    code_execution_config=False,
    default_auto_reply="",
    is_termination_msg=lambda x: True,
)